# SAMMD canonical lightweight workflow

This notebook exercises the current SAMMD MVP without OpenMM, OpenFF, RDKit, mBuild, or long computations. It validates a YAML template, builds a deterministic lightweight plan, writes the current build artifacts, and inspects those outputs.

Current limitation: `build_system()` returns a planning object, not a full OpenMM simulation system. The current build writes `topology.cif`, `build_summary.json`, and `resolved_config.yaml`; future backend exports reserve `positions.cif`, `interchange.json`, `system.xml`, and `anchor_metadata.json`.

## Create a template configuration

The CLI command `sammd init -o sammd.yaml` writes the same template. Here we write it directly so the notebook stays self-contained.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from sammd.config import CONFIG_TEMPLATE

workspace = TemporaryDirectory()
workdir = Path(workspace.name)
config_path = workdir / "sammd.yaml"
output_dir = workdir / "outputs"
config_path.write_text(CONFIG_TEMPLATE, encoding="utf-8")

print(f"Wrote template config to {config_path}")

## Validate, load, and build a plan

`load_config()` validates YAML through the Pydantic schema. `build_system()` composes the current deterministic planning artifact.

In [ ]:
from sammd import build_system, load_config

config = load_config(config_path)
plan = build_system(config, output_dir=output_dir)

print("Configuration validated")
print(f"Full OpenMM construction available: {plan.full_construction_available}")

## Write current artifacts

The current lightweight build explicitly writes exactly three artifacts: `topology.cif`, `build_summary.json`, and `resolved_config.yaml`.

In [ ]:
topology_path = plan.write_topology_cif(overwrite=True)
build_summary_path = plan.write_build_summary(overwrite=True)
resolved_config_path = plan.write_resolved_config(overwrite=True)

for path in (topology_path, build_summary_path, resolved_config_path):
    print(f"Wrote {path.name}: {path}")

## Inspect current outputs

These values are useful for checking the planned surface, adjusted commensurate dimensions, SAM occupancy, solution counts, and current output files. Open `topology.cif` in a molecule viewer such as PyMOL to inspect the configured surface and SAM sulfur anchor placeholders.

In [ ]:
summary = {
    "surface": f"{plan.slab.metal}({plan.slab.facet})",
    "adjusted_lateral_size_nm": tuple(round(value, 3) for value in plan.slab.lateral_size_nm),
    "binding_site_count": len(plan.binding_sites),
    "sam_placement_count": len(plan.sam_placements.placements),
    "sam_sites_per_side": plan.sam_placements.selected_sites_per_side,
    "solution_counts": plan.solution.molecule_counts,
}

for key, value in summary.items():
    print(f"{key}: {value}")

print()
for path in (topology_path, build_summary_path, resolved_config_path):
    print(f"{path.name}: exists={path.exists()} size={path.stat().st_size} bytes")

## Reserved future backend artifacts

The current release does not write `positions.cif`, `interchange.json`, `system.xml`, or `anchor_metadata.json`. Those names may appear in resolved paths or planning metadata, but they are reserved target artifacts, not current outputs.

Future backend construction is expected to use `interchange.json` as the primary portable OpenFF Interchange export, written with `Interchange.model_dump_json` and reloaded with `Interchange.model_validate_json`. Pre-1.0 Interchange JSON compatibility is not guaranteed across OpenFF Interchange versions.

## Future OpenMM handoff

After a future SAMMD backend writes `interchange.json` and companion artifacts, students will hand those build artifacts to their own OpenMM Python API script. The intended teaching path is to reload the portable OpenFF Interchange data from `interchange.json` with `Interchange.model_validate_json`, export an OpenMM `System` from that Interchange object with `interchange.to_openmm()`, load constructed coordinates from `positions.cif`, optionally use `system.xml` only as a convenience OpenMM system export, and create a raw OpenMM `Simulation` directly through the OpenMM Python API. SAMMD does not provide OpenMM simulation wrappers for this handoff.

That handoff is not runnable in this lightweight release because `positions.cif`, `interchange.json`, `system.xml`, and `anchor_metadata.json` are reserved target artifacts, not current outputs. The important boundary is unchanged: SAMMD builds; OpenMM runs.

## Cleanup

The temporary directory can be removed when you are done inspecting generated files. Comment this out if you want to keep `topology.cif`.

In [ ]:
workspace.cleanup()